# Phase 1: Data Extraction

## 1. Imports

In [9]:
from nba_api.stats.endpoints import leaguegamefinder
import pandas as pd
import time
from pathlib import Path

## 2. Extract Game Data (2024-25, 2025-26 — Regular Season & Playoffs)

**Question:** What does our raw game-level dataset look like across the two 
training seasons, and does the row count match expectations?

We pull both season types separately per season, using `nba_api`'s live 
endpoint. Regular season pulls already include NBA Cup group stage and 
knockout games natively (confirmed via investigation — the NBA Cup 
Championship Final is the only game tagged separately, and it's excluded 
by default since it doesn't fall under 'Regular Season').

In [10]:
seasons = ['2024-25', '2025-26']
season_types = ['Regular Season', 'Playoffs']

all_games = []

for season in seasons:
    for stype in season_types:
        gf = leaguegamefinder.LeagueGameFinder(
            season_nullable=season,
            season_type_nullable=stype
        )
        df = gf.get_data_frames()[0]
        df['SEASON_LABEL'] = season
        df['SEASON_TYPE_LABEL'] = stype
        all_games.append(df)
        time.sleep(1)  # To avoid hitting the API rate limit

games_combined =pd.concat(all_games, ignore_index=True)

In [11]:
games_combined.groupby(['SEASON_LABEL', 'SEASON_TYPE_LABEL']).size()

SEASON_LABEL  SEASON_TYPE_LABEL
2024-25       Playoffs              168
              Regular Season       2460
2025-26       Playoffs              170
              Regular Season       2460
dtype: int64

**Takeaway:** Both the 2024-25 and 2025-26 regular seasons return the full 2,460 games, 
confirming both seasons are fully complete. Playoffs total 168 games in 
2024-25 and 170 in 2025-26 — the difference reflects series length 
variation (more games going to 6 or 7 games in 2025-26).


## 3. Sanity Check Before Saving

**Question:** Does anything look wrong before we lock this in as our 
clean dataset?

A quick check for nulls or unexpected duplicate rows protects the 
integrity of every feature built downstream in Phase 2.

In [12]:
# checking for null values across all columns
games_combined.isnull().sum()

# checking for duplicate rows
games_combined.duplicated().sum()

# confirm season type and season label columns are correct
games_combined['SEASON_TYPE_LABEL'].unique()

<StringArray>
['Regular Season', 'Playoffs']
Length: 2, dtype: str

**Takeaway:** The data cleaning verification confirms the dataset is structured correctly and ready for analysis:

*   **No Missing Data**: There are zero null values present across all columns in the dataset.
    
*   **No Duplicate Rows**: The row-level integrity is maintained with zero duplicate records found.
    
*   **Consistent Categories**: The `SEASON_TYPE_LABEL` column contains only the expected `Regular Season` and `Playoffs` values, with no anomalies or typos.

## 4. Save Cleaned Dataset

Exporting the verified game-level dataset to `data/processed/` as a 
clean, versioned handoff to Phase 2 (feature engineering).

In [16]:
# save the combined dataframe to a CSV file to data/processed/
games_combined.to_csv(r"D:\Data Science Projects\nba-26-27-predictor\data\processed\games_clean.csv", index=False)